In [ ]:
#This code helps to oversample the dataset

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RepeatedKFold
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

x=np.load('xlo_1.npy')
y=np.load('ylo.npy')
y[:,1]=y[:,1]+y[:,3] 
y[:,7]=y[:,7]+y[:,2] #Other=HCP+Liquid
phs=7 #phase ( fcc= 0, bcc = 1, laves = 4, Sigma = 5, Heusler = 6, Liquid = 7)
cv = RepeatedKFold(n_splits=10, n_repeats=1, random_state=1)  
cv_num=0
for train_ix, test_ix in cv.split(x):
        x_lo_train, x_lo_val = x[train_ix], x[test_ix]
        y_lo_train, y_lo_val = y[train_ix], y[test_ix]    
        if cv_num==0:
            break            
rng = 0
with open('xlo_val.npy','wb') as f:
    np.save(f,x_lo_val)
with open('ylo_val.npy','wb') as f:
    np.save(f,y_lo_val)       
x=np.copy(x_lo_train)
y=np.copy(y_lo_train)
bin_range = (0, 1)
bin_size = 0.1
num_bins = int((bin_range[1] - bin_range[0]) / bin_size)
bins1 = np.arange(bin_range[0], bin_range[1] + bin_size, bin_size)
bins1=np.round(bins1, decimals=5)
ZZ=plt.hist(y[:,phs],bins=bins1) 


n=[]
for i in range(y.shape[0]):
    if y[i,phs]>=0.1: #Note: either < 0.9 or >= 0.1 (default) which is based on the assumption that [0,0.1] is major 
        n=np.append(n,i)
x1=np.delete(x,n.astype('int'),axis=0)
y1=np.delete(y,n.astype('int'),axis=0)
with open('xlo_mindel.npy','wb') as f:  
    np.save(f,x1)
with open('ylo_mindel.npy','wb') as f: 
    np.save(f,y1)
n=[]
for i in range(y.shape[0]):
    if y[i,phs]<0.1: #Note: either >=0.9 or <0.1 
        n=np.append(n,i)
x1=np.delete(x,n.astype('int'),axis=0)
y1=np.delete(y,n.astype('int'),axis=0)
with open('xlo_majdel.npy','wb') as f:
    np.save(f,x1)
with open('ylo_majdel.npy','wb') as f:
    print('n=',n.shape[0])
    np.save(f,y1)
count_zero=round(n.shape[0]*0.5) #i.e. 50% of the largest bin


A = np.load('ylo_majdel.npy')[:, phs:phs+1].flatten()
x = np.load('xlo_majdel.npy') 
bin_range = (0, 1)
bin_size = 0.1
num_bins = int((bin_range[1] - bin_range[0]) / bin_size)
bins1 = np.arange(bin_range[0], bin_range[1] + bin_size, bin_size)
bins1=np.round(bins1, decimals=5)
ZZ=plt.hist(A,bins=bins1) 
bin_counts=ZZ[0][1:]
bin_range=ZZ[1][1:]
oversampled_A = []
oversampled_x = []
i=0
stop=0
for i in range(9):
    if bin_counts[i]==0:
        pass
    else:
        print('count=',bin_counts[i])
        if bin_counts[i] < count_zero:
            factor = np.ceil(count_zero / bin_counts[i]).astype(int)
            print('factor=',factor)
            if i!=8:
                indices_in_bin = np.where((A >= bin_range[i]) & (A < bin_range[i+1]))[0] 
            else:
                indices_in_bin = np.where((A >= bin_range[i]) & (A <= bin_range[i+1]))[0]   
            data_A_in_bin = A[indices_in_bin]
            data_x_in_bin = x[indices_in_bin]
            oversampled_A.append(np.tile(data_A_in_bin, factor))
            oversampled_x.append(np.tile(data_x_in_bin, (factor, 1)))            
        else:
            factor=1
            indices_in_bin = np.where((A >= bin_range[i]) & (A < bin_range[i+1]))[0] 
            data_A_in_bin = A[indices_in_bin]
            data_x_in_bin = x[indices_in_bin]
            oversampled_A.append(np.tile(data_A_in_bin, factor))
            oversampled_x.append(np.tile(data_x_in_bin, (factor, 1)))
oversampled_A = np.concatenate(oversampled_A)
oversampled_x = np.concatenate(oversampled_x)
shuffle_indices = np.random.permutation(len(oversampled_A))
oversampled_A = oversampled_A[shuffle_indices]
oversampled_x = oversampled_x[shuffle_indices]
Ax=np.load('xlo_mindel.npy')
Ay=np.load('ylo_mindel.npy')[:,phs:phs+1] 
Ax=np.append(Ax,oversampled_x,axis=0)
Ay=np.append(Ay,oversampled_A.reshape(-1,1),axis=0)


bin_range = (0, 1)
bin_size = 0.1
num_bins = int((bin_range[1] - bin_range[0]) / bin_size)
bins1 = np.arange(bin_range[0], bin_range[1] + bin_size, bin_size)
bins1=np.round(bins1, decimals=5)
ZZ=plt.hist(Ay,bins=bins1)
plt.xlabel('Liquid phase fraction')
plt.ylabel('Count')
ZZ


with open('xlo_oversampling.npy','wb') as f:
    np.save(f,Ax)
with open('ylo_oversampling.npy','wb') as f:
    np.save(f,Ay)